# 第13回 なぜ間違えるのか — ハルシネーションと、AI が「知らない」と言えない理由

**このノートブックは第13回だけで完結します。** プログラミングの経験は要りません。

やること:
1. **存在しない国の首都を聞いて、モデルの「自信」を測る**
2. 自信の高さ別に正解率を出し、自信があてになるかを確かめる
3. 間違え方に「型」があることを見る
4. **何度も聞いて、知らないことを見抜く方法を試す**

所要時間:実習 約25分 / 課題 約40分

> **注意**:3GBほどのモデルをダウンロードします。時間に余裕をもって実行してください。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

> **注意**:3GBほどのモデルをダウンロードします。最初のセルで3〜5分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。3〜5分かかります)
!pip install -q transformers japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np
from collections import Counter
import matplotlib.pyplot as plt, japanize_matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
plt.rcParams["font.size"] = 12
トークナイザ = AutoTokenizer.from_pretrained("gpt2")
モデル = AutoModelForCausalLM.from_pretrained("gpt2-large"); モデル.eval()

def 予測(文, 上位=3):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(モデル(ids).logits[0, -1], dim=-1)
    上 = torch.topk(p, 上位)
    return [(トークナイザ.decode([i]).strip() or "␣", float(v) * 100)
            for i, v in zip(上.indices, 上.values)]

print("準備できました")

## 1. 知らないことを、知らないと言えるか

存在しない国の名前を8つ用意しました(すべて架空です)。その首都を聞きます。
**答えようがない質問**です。

モデルの「自信」── 1位の語の確率 ── は、実在する国のときと比べてどうなるでしょうか。
**予想してから実行してください。**

In [ ]:
# ===== ① 知らないことを、知らないと言えるか =====
# 存在する国と、私が今日でっちあげた国名を並べて聞きます。
実在する国 = ["France", "Japan", "Italy", "Egypt", "Spain", "Canada", "Brazil", "Norway"]
架空の国 = ["Zeldoria", "Marnovia", "Thessaly Prime", "Kelmoria",
            "Vantissia", "Blaythorne", "Aoberia", "Midoria"]

def 首都を聞く(国):
    return 予測(f"Q: What is the capital of {国}? A:")

print("【実在する国】")
実在の自信 = []
for 国 in 実在する国:
    r = 首都を聞く(国); 実在の自信.append(r[0][1])
    print(f"  {国:<16}→ {r[0][0]:<12}{r[0][1]:5.1f}%")
print(f"  1位の確率の平均 {np.mean(実在の自信):.1f}%\n")

print("【存在しない国(すべて架空の名前)】")
架空の自信 = []
for 国 in 架空の国:
    r = 首都を聞く(国); 架空の自信.append(r[0][1])
    print(f"  {国:<16}→ {r[0][0]:<12}{r[0][1]:5.1f}%")
print(f"  1位の確率の平均 {np.mean(架空の自信):.1f}%\n")

print(f"実在 {np.mean(実在の自信):.1f}%  vs  架空 {np.mean(架空の自信):.1f}%"
      f"   差 {np.mean(実在の自信)-np.mean(架空の自信):.1f} ポイント")
print()
最高 = 架空の国[int(np.argmax(架空の自信))]
print(f"架空の「{最高}」への自信 {max(架空の自信):.1f}% は、")
print(f"実在する国の最高 {max(実在の自信):.1f}% と比べてどうでしょうか。")

In [ ]:
# 図で見る
plt.figure(figsize=(9, 4))
x1 = np.arange(len(実在する国)); x2 = np.arange(len(架空の国)) + len(実在する国) + 1
plt.bar(x1, 実在の自信, color="#3A46A8", label="実在する国")
plt.bar(x2, 架空の自信, color="#E08A2B", label="存在しない国(架空)")
plt.axhline(np.mean(実在の自信), color="#3A46A8", ls="--", lw=1)
plt.axhline(np.mean(架空の自信), color="#E08A2B", ls="--", lw=1)
plt.xticks(list(x1) + list(x2), 実在する国 + 架空の国, rotation=60, ha="right", fontsize=9)
plt.ylabel("1位の語の確率(%)"); plt.legend(); plt.grid(alpha=.3, axis="y")
plt.title("モデルは「知らない」を自分で分かっているか")
plt.tight_layout(); plt.show()

print("2本の破線(それぞれの平均)が、ほとんど重なっていませんか。")
print("→ モデルは「知っていること」と「そもそも存在しないこと」を区別できていません。")

In [ ]:
# 実際に文章を書かせてみる
for 文 in ["The capital of Zeldoria is",
           "The population of Marnovia is about",
           "Kelmoria was founded in the year"]:
    ids = トークナイザ.encode(文, return_tensors="pt")
    torch.manual_seed(1)
    with torch.no_grad():
        出力 = モデル.generate(ids, max_new_tokens=20, do_sample=True, temperature=0.7,
                               top_k=50, pad_token_id=トークナイザ.eos_token_id,
                               attention_mask=torch.ones_like(ids))
    print("  " + トークナイザ.decode(出力[0], skip_special_tokens=True).replace("\n", " "))
    print()

print("存在しない国について、具体的な数字や歴史を書いていませんか。")
print("ためらいも、但し書きもありません。これがハルシネーションです。")
print()
print("【大事なこと】嘘をつこうとしているのではありません。")
print("いつもどおり「もっともらしい続き」を書いただけです。")
print("正解が出るときも、でたらめが出るときも、内部の動作はまったく同じです。")

> **なぜ「知りません」と言えないのか**
>
> 1. **事実の台帳がない** … モデルの中に「知っていること一覧」はありません。あるのは重みの数字だけです
> 2. **必ず何かを出す作り** … 出力は語ごとの確率で、合計は必ず 1。「該当なし」という出口がありません
> 3. **分からないと言う練習が足りない** … 学習データに「私は知りません」と続く文は、めったにありません
>
> つまりハルシネーションは**故障ではなく、この仕組みの正常な出力**です。

## 2. では、自信はまったくあてにならないのか

30問(首都・簡単な計算・常識)を出して、答えを正誤で判定し、
**自信の高さ別に正解率**を見てみます。

In [ ]:
# ===== ② 自信は、あてになるのか(較正)=====
問題 = [
    ("Q: What is the capital of France? A:", "Paris"),
    ("Q: What is the capital of Japan? A:", "Tokyo"),
    ("Q: What is the capital of Italy? A:", "Rome"),
    ("Q: What is the capital of Spain? A:", "Madrid"),
    ("Q: What is the capital of Egypt? A:", "Cairo"),
    ("Q: What is the capital of Canada? A:", "Ottawa"),
    ("Q: What is the capital of Norway? A:", "Oslo"),
    ("Q: What is the capital of Brazil? A:", "Bras"),
    ("Q: What is the capital of Greece? A:", "Athens"),
    ("Q: What is the capital of Russia? A:", "Moscow"),
    ("Q: What is the capital of China? A:", "Beijing"),
    ("Q: What is the capital of India? A:", "New"),
    ("Q: What is the capital of Kenya? A:", "Nairobi"),
    ("Q: What is the capital of Peru? A:", "Lima"),
    ("Q: What is the capital of Turkey? A:", "Ankara"),
    ("Q: What is the capital of Poland? A:", "Warsaw"),
    ("Q: What is the capital of Sweden? A:", "Stockholm"),
    ("Q: What is the capital of Austria? A:", "Vienna"),
    ("Q: What is the capital of Portugal? A:", "Lisbon"),
    ("Q: What is the capital of Ireland? A:", "Dublin"),
    ("Q: What is the capital of Thailand? A:", "Bangkok"),
    ("Q: What is the capital of Vietnam? A:", "Hanoi"),
    ("Q: What is the capital of Chile? A:", "Santiago"),
    ("Q: What is the capital of Cuba? A:", "Havana"),
    ("Q: What is 2 plus 2? A:", "4"),
    ("Q: What is 5 plus 3? A:", "8"),
    ("Q: What color is the sky? A:", "blue"),
    ("Q: How many days are in a week? A:", "7"),
    ("Q: What is the opposite of hot? A:", "cold"),
    ("Q: How many legs does a spider have? A:", "8"),
]

結果 = []
for q, 正答 in 問題:
    語, 確率 = 予測(q, 1)[0]
    結果.append((確率, 語.lower() == 正答.lower(), 語, 正答, q))
結果.sort(key=lambda r: -r[0])

print(f"{'自信':>7}  {'AIの答え':<12}{'正解':<12}判定")
print("-" * 46)
for 確率, 正, 語, 正答, q in 結果:
    print(f"{確率:>6.1f}%  {語:<12}{正答:<12}{'○' if 正 else '×'}")
print(f"\n全体の正解率 {sum(r[1] for r in 結果)/len(結果)*100:.1f}%"
      f"  ({sum(r[1] for r in 結果)}/{len(結果)})")

In [ ]:
# 自信の高さ別に、正解率を見る
区分 = [(0, 20), (20, 30), (30, 50), (50, 100)]
表 = []
print(f"{'自信の範囲':<14}{'問題数':>8}{'正解数':>8}{'正解率':>9}{'平均の自信':>11}")
print("-" * 52)
for lo, hi in 区分:
    組 = [r for r in 結果 if lo <= r[0] < hi]
    if not 組: continue
    正解数 = sum(r[1] for r in 組)
    表.append((f"{lo}〜{hi}%", len(組), 正解数, 正解数/len(組)*100, np.mean([r[0] for r in 組])))
    print(f"{lo:>3}〜{hi:>3}%{'':<6}{len(組):>7}問{正解数:>7}問{正解数/len(組)*100:>8.1f}%"
          f"{np.mean([r[0] for r in 組]):>10.1f}%")

plt.figure(figsize=(7.5, 4))
plt.bar([r[0] for r in 表], [r[3] for r in 表], color="#3A46A8")
plt.plot([r[0] for r in 表], [r[4] for r in 表], "o--", color="#E08A2B", label="平均の自信")
plt.ylabel("%"); plt.legend(); plt.grid(alpha=.3, axis="y")
plt.title("青=実際の正解率 / オレンジ=モデルの自信")
plt.tight_layout(); plt.show()

print("自信が上がるほど、正解率も上がっているはずです。")
print("つまり「自信が低いときは疑う」は、それなりに有効な指針です。")
print()
print("ただし2つ注意:")
print("① 自信の数値は確率として読めません(自信33%でも実際は8割方正解)。順序の情報です。")
print("② この表は『答えが存在する質問』でしか作れません。架空の国の首都には正解がないので、")
print("   正解率も出せません。そして、そこでの自信は実在の国とほとんど同じでした。")

## 3. 間違え方には、型がある

間違えた問題を、自信が高い順に並べてみます。
**でたらめな誤り**でしょうか。それとも、何か規則があるでしょうか。

In [ ]:
# ===== ③ 間違え方には、型がある =====
誤答 = [r for r in 結果 if not r[1]]
print("間違えた問題(自信が高い順)")
print("-" * 64)
for 確率, _, 語, 正答, q in 誤答:
    print(f"{確率:>6.1f}%  {q}")
    print(f"          AI:「{語}」  正解:「{正答}」")
print()
print("でたらめな語ではなく、『惜しい』誤りになっていませんか。")
print("  Turkey  → Istanbul(最大都市。首都はアンカラ)")
print("  Brazil  → Rio(有名な都市。首都はブラジリア)")
print()
print("第7回で見たとおり、モデルは『一緒に出てくる語』を学んでいます。")
print("トルコとイスタンブールは文章の中で何度も並びます ── 首都かどうかとは無関係に。")
print("事実を照合しているのではなく、結びつきの強い語を選んでいるのです。")
print()
print("もっともらしいからこそ、見逃されます。知らない人には正解に見えてしまう。")

## 4. どうすれば見抜けるか — 何度も聞いてみる

考え方は単純です ──
**本当に知っていることなら、何度聞いても同じ答えが返るはずです。**

温度を上げて(毎回少し揺らして)20回引き、答えがどれくらい揃うかを測ります。

In [ ]:
# ===== ④ どうすれば見抜けるか — 何度も聞いてみる =====
def 何回も聞く(文, 回数=20, 温度=0.8, seed=0):
    """温度を上げて何度も引き、答えがどれくらい揃うかを見る"""
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        logits = モデル(ids).logits[0, -1]
    p = torch.softmax(logits / 温度, dim=-1)
    g = torch.Generator().manual_seed(seed)
    引いた = torch.multinomial(p, 回数, replacement=True, generator=g)
    語 = [トークナイザ.decode([int(i)]).strip() or "␣" for i in 引いた]
    c = Counter(語)
    最多語, 最多数 = c.most_common(1)[0]
    return 最多語, 最多数 / 回数 * 100, len(c), c

検査 = [("実在", "Austria"), ("実在", "France"), ("実在", "Japan"), ("実在", "Sweden"),
        ("架空", "Zeldoria"), ("架空", "Marnovia"), ("架空", "Kelmoria"), ("架空", "Vantissia")]
実在割合, 架空割合, 実在種類, 架空種類 = [], [], [], []

for 種別, 国 in 検査:
    語, 割合, 種類, c = 何回も聞く(f"Q: What is the capital of {国}? A:")
    (実在割合 if 種別 == "実在" else 架空割合).append(割合)
    (実在種類 if 種別 == "実在" else 架空種類).append(種類)
    上位 = ", ".join(f"{k}×{v}" for k, v in c.most_common(3))
    print(f"[{種別}] {国:<12} 最多「{語}」{割合:>3.0f}%  {種類:>2}種類   {上位}")

print()
print(f"実在する国  : 最多の割合 平均 {np.mean(実在割合):.0f}%   出た語 平均 {np.mean(実在種類):.1f} 種類")
print(f"存在しない国: 最多の割合 平均 {np.mean(架空割合):.0f}%   出た語 平均 {np.mean(架空種類):.1f} 種類")

In [ ]:
# 図で見る
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(["実在する国", "存在しない国"], [np.mean(実在割合), np.mean(架空割合)],
          color=["#3A46A8", "#E08A2B"])
ax[0].set_ylabel("最多の答えが占める割合(%)"); ax[0].grid(alpha=.3, axis="y")
ax[0].set_title("答えは揃うか")
ax[1].bar(["実在する国", "存在しない国"], [np.mean(実在種類), np.mean(架空種類)],
          color=["#3A46A8", "#E08A2B"])
ax[1].set_ylabel("20回で出てきた語の種類"); ax[1].grid(alpha=.3, axis="y")
ax[1].set_title("答えはばらつくか")
plt.tight_layout(); plt.show()

print("【皆さんが今日から使える方法】")
print("大事な答えは、新しい会話を開いて、言い方を変えて3回聞いてください。")
print("  ・毎回同じ → まだ信用してよい可能性がある")
print("  ・ばらつく → ほぼ確実に、根拠がありません")
print()
print("1回の自信より、3回の一貫性のほうが役に立ちます。")
print()
print("【ただし限界もあります】")
print("『一貫した誤り』は、この方法でも見抜けません。")
print("トルコの首都をイスタンブールと答えるのは、何度聞いても同じはずです。")
print("→ これは「ばらつけば怪しい」という片側だけの判定です。")

> **「合っていますか」と聞き返すのは、確認になりません**
>
> 第9回でみた RLHF により、モデルは「人が good と言いそうな答え」に調整されています。
> 利用者が不満を示すと、**同意して訂正するほうが好まれやすい** ── だから押せば折れます(**おべっか**)。
>
> | 効かない | 効く |
> |---|---|
> | 「合っていますか」と聞き返す | 新しい会話で、言い方を変えて3回聞く |
> | 「自信はありますか」と聞く | 一次資料で確かめる(公式サイト・書籍) |
> | 同じ会話でもう一度聞く | 資料を自分で貼って答えさせる(第11回) |
>
> 効く方法はどれも、**いったん AI の外に出る作業**だという点が共通しています。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分で架空の名前を5つ作って、実在するものと比べる
私の架空 = ["Norvania", "Suzuran Republic", "Aoba Federation", "Midorigaoka", "Kelvaria"]
私の実在 = ["Germany", "Mexico", "Kenya", "Finland", "Chile"]

print(f"{'名前':<20}{'AIの答え':<14}{'自信'}")
print("-" * 44)
架空自信, 実在自信 = [], []
for 国 in 私の実在:
    r = 首都を聞く(国); 実在自信.append(r[0][1])
    print(f"[実在] {国:<14}{r[0][0]:<14}{r[0][1]:5.1f}%")
for 国 in 私の架空:
    r = 首都を聞く(国); 架空自信.append(r[0][1])
    print(f"[架空] {国:<14}{r[0][0]:<14}{r[0][1]:5.1f}%")
print(f"\n実在 {np.mean(実在自信):.1f}%  vs  架空 {np.mean(架空自信):.1f}%"
      f"   差 {np.mean(実在自信)-np.mean(架空自信):.1f} ポイント")

# (b) 自分の質問で、ばらつきを測る
print("\n" + "=" * 50)
for q in ["Q: What is the capital of Finland? A:",
          "Q: What is the capital of Norvania? A:"]:
    語, 割合, 種類, c = 何回も聞く(q)
    print(f"{q}")
    print(f"   最多「{語}」{割合:.0f}%   {種類}種類   {dict(c.most_common(4))}")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:AI の答えを検討する
```
【質問】「カナダの首都はどこですか」
【答え】「カナダの首都はトロントです。人口約280万人で、同国最大の都市です。」
```

- (a) この答えは正しいですか
- (b) 今日見たどの「型」にあてはまりますか
- (c) 「人口約280万人」という部分は、誤りを見抜き**やすく**していますか、**にくく**していますか
- (d) この答えを受け取った人が、提出前にすべきことを2つ挙げなさい

### 課題2:ノートブックを動かして報告する
- (a) 自分で作った架空の名前5つと、実在するもの5つの自信の比較表
- (b) 自信の高さ別の正解率の表と、**高い自信で間違えた問題の具体例**
- (c) 同じ質問を20回聞いたときのばらつき(実在 vs 架空)

> (b) は表だけでは評価を下げます。**実物の誤答を挙げてください。**それが今日いちばんの発見です。

### 課題3:次の主張を批判する(300字以上)

> 「ハルシネーションは技術的な不具合です。モデルを改良すれば、いずれ完全になくなります。
> それまでは、AI に『自信はありますか』と確認すれば十分です。」

考える材料:
- 「不具合」と呼べますか。正解を出すときと誤るとき、モデルの動作は違いますか
- 「いずれ完全になくなる」の根拠はありますか。今日の実測のどれが反証になりますか
- 「自信はありますか」と聞くのは、なぜ確認にならないのですか(**理由は2つあります**)
- では、実際に有効な手段は何ですか。それらに共通することは何ですか

**加点**:「複数回聞いても見抜けない誤りがある」という限界まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。